# SPT-3G BB Analysis — Initial Exploration

Local development notebook for SPT-3G CMB polarization analysis.  
Real data lives on the cluster; the hardcoded paths in §1 are cluster-only.

In [ ]:
import os, sys
import numpy as np
import healpy as hp
import matplotlib
import matplotlib.pyplot as plt
from spt3g import core, maps   # maps must be imported to register HealpixSkyMap

from Plot import (
    apply_spt_style,
    mask_map,
    show_map_full_field,
    show_map_thumbnail,
    show_1d_functions_of_ell,
    show_alm_triangle,
    MapPlotter,
)

# Apply SPT-3G standard plot style for the whole notebook
apply_spt_style()

print("python     :", sys.version)
print("numpy      :", np.__version__)
print("healpy     :", hp.__version__)
print("matplotlib :", matplotlib.__version__)

## 1. Data Discovery

> **Cluster only.** These cells explore the raw data directories and will not run locally.

In [ ]:
base_sptgrid = "/sptgrid/data/"
for name in sorted(os.listdir(base_sptgrid)):
    path = os.path.join(base_sptgrid, name)
    try:
        children = sorted(os.listdir(path))
        print(f"[{name}]  ({len(children)} items)")
        for c in children[:5]:
            print(f"  {c}")
    except (NotADirectoryError, PermissionError) as e:
        print(f"[{name}]  ({type(e).__name__})")


[3g_verified]  (0 items total)



[arc]  (587624 items total)
  .20180303_165838.dat.J6myqQ
  .20180304_161838.dat.7eLzLi
  .20180305_071838.dat.zFMQ8Q
  .20180305_120158.dat.qGCfJV
  .20180305_150518.dat.Fs0FK7

[balloon_gps_data]  (5 items total)
  2017
  2018
  2019
  2020
  2021

[bolodata]  (2 items total)
  downsampled
  fullrate

[onlinemaps]  (63 items total)
  J1745-290
  calframe
  calibration
  lightcurve
  monthly_coadds

[sptpol]  (34 items total)
  (none)
  0521-365
  0537-441
  2255-282
  CenA


> **File structure:** Under `bb2019/full_2019_and_2020/` each subfield has a `pol_sub_optimized_healpix/` directory. Inside, each OBSID has its own sub-directory containing a single `.g3` file named with that OBSID.

In [ ]:
base_bb2019 = "/sptlocal/analysis/bb2019/full_2019_and_2020"
print("Subfields:", sorted(os.listdir(base_bb2019)))

Number of items: 4
['ra0hdec-44.75', 'ra0hdec-52.25', 'ra0hdec-59.75', 'ra0hdec-67.25']


In [ ]:
subfield = "ra0hdec-44.75"
obsid    = "118022023"

map_file = (
    f"{base_bb2019}/{subfield}/pol_sub_optimized_healpix/"
    f"{obsid}/pol_sub_optimized_healpix_{obsid}.g3"
)
print(map_file)
print("Exists:", os.path.exists(map_file))

/sptlocal/analysis/bb2019/full_2019_and_2020/ra0hdec-44.75/pol_sub_optimized_healpix/118022023/pol_sub_optimized_healpix_118022023.g3
True


## 2. Map Inspection

In [ ]:
for i, fr in enumerate(core.G3File(map_file)):
    if fr.type == core.G3FrameType.Map:
        print("Map frame index:", i)
        print("Id:", fr["Id"])
        print("Keys:", list(fr.keys()))
        for k in fr.keys():
            print(" ", k, type(fr[k]))
        break

Map frame: 6
Keys: ['Q', 'U', 'Id', 'T', 'Wpol']
Q <class 'spt3g.maps.HealpixSkyMap'>
U <class 'spt3g.maps.HealpixSkyMap'>
Id <class 'str'>
T <class 'spt3g.maps.HealpixSkyMap'>
Wpol <class 'spt3g.maps.G3SkyMapWeights'>


In [ ]:
for fr in core.G3File(map_file):
    if fr.type == core.G3FrameType.Map:
        print("Map Id:", fr["Id"])
        for key in ["T", "Q", "U"]:
            arr  = np.asarray(fr[key])
            good = np.isfinite(arr)
            print(f"\n--- {key} ---")
            print("  nside  :", fr[key].nside)
            print("  units  :", fr[key].units)
            print("  finite :", good.sum())
            print("  min/max/mean:", arr[good].min(),
                                    arr[good].max(),
                                    arr[good].mean())
        break

Map Id: Left90GHz



--- T ---
type: <class 'spt3g.maps.HealpixSkyMap'>
nside: 2048
coord_ref: Equatorial
units: Tcmb
npix: 50331648
finite pixels: 50331648
min: -12058798591.083391
max: 75350763352.9389
mean: 74646.06820164238

--- Q ---
type: <class 'spt3g.maps.HealpixSkyMap'>
nside: 2048
coord_ref: Equatorial
units: Tcmb
npix: 50331648
finite pixels: 50331648
min: -2990009920.075752
max: 3028386617.1435046
mean: -3180.1537935418723

--- U ---
type: <class 'spt3g.maps.HealpixSkyMap'>
nside: 2048
coord_ref: Equatorial
units: Tcmb
npix: 50331648
finite pixels: 50331648
min: -2964661609.614376
max: 3106291755.968077
mean: -1206.8947176943454


In [ ]:
# Weight map (Wpol) components
for fr in core.G3File(map_file):
    if fr.type == core.G3FrameType.Map:
        w = fr["Wpol"]
        for comp in ["TT", "TQ", "TU", "QQ", "QU", "UU"]:
            print(comp, type(getattr(w, comp)))
        break

TT <class 'spt3g.maps.HealpixSkyMap'>
TQ <class 'spt3g.maps.HealpixSkyMap'>
TU <class 'spt3g.maps.HealpixSkyMap'>
QQ <class 'spt3g.maps.HealpixSkyMap'>
QU <class 'spt3g.maps.HealpixSkyMap'>
UU <class 'spt3g.maps.HealpixSkyMap'>


In [ ]:
# All Map frames contained in the .g3 file
for i, fr in enumerate(core.G3File(map_file)):
    if fr.type == core.G3FrameType.Map:
        print(i, fr["Id"])

6 Left90GHz ['Q', 'U', 'Id', 'T', 'Wpol']
7 Left150GHz ['Q', 'U', 'Id', 'T', 'Wpol']
8 Left220GHz ['Q', 'U', 'Id', 'T', 'Wpol']
9 Right90GHz ['Q', 'U', 'Id', 'T', 'Wpol']
10 Right150GHz ['Q', 'U', 'Id', 'T', 'Wpol']
11 Right220GHz ['Q', 'U', 'Id', 'T', 'Wpol']


## 3. Load Map Data

Load all six frequency/side frames into a `frames` dict that `MapPlotter` reads directly. Also define the subfield centre and the 3 × 2 display layout.

In [ ]:
target_ids = [
    "Left90GHz",  "Right90GHz",
    "Left150GHz", "Right150GHz",
    "Left220GHz", "Right220GHz",
]

frames = {}
for fr in core.G3File(map_file):
    if fr.type != core.G3FrameType.Map:
        continue
    mid = fr["Id"]
    if mid in target_ids:
        W = fr["Wpol"]
        frames[mid] = {
            "T":  np.asarray(fr["T"],  float),
            "Q":  np.asarray(fr["Q"],  float),
            "U":  np.asarray(fr["U"],  float),
            "TT": np.asarray(W.TT, float),
            "QQ": np.asarray(W.QQ, float),
            "UU": np.asarray(W.UU, float),
        }

print(f"Loaded {len(frames)} frames:", list(frames))

In [ ]:
# Subfield centre [RA, Dec] degrees — change for each subfield
rot_centre = [0, -44.75]

# 3x2 display layout: rows = frequency, columns = detector side
layout = [
    ("Left90GHz",  "Right90GHz"),
    ("Left150GHz", "Right150GHz"),
    ("Left220GHz", "Right220GHz"),
]

In [ ]:
# Pull arrays from one frame for single-map plots and E/B analysis
ref_id = "Left90GHz"
T  = frames[ref_id]["T"]
Q  = frames[ref_id]["Q"]
U  = frames[ref_id]["U"]
TT = frames[ref_id]["TT"]
QQ = frames[ref_id]["QQ"]
UU = frames[ref_id]["UU"]

## 4. Visualisation

All plotting uses `MapPlotter` from `Plot.py`. Two backends: `gnomview` (healpy) and `imshow` (matplotlib via `GnomonicProj`).

In [ ]:
# Weight map key helper (used in §4 cells)
_wt = {"T": "TT", "Q": "QQ", "U": "UU"}.get

plotter = MapPlotter(frames)   # kept for multi-panel grid plots

### 4.1 Single-map preview — gnomview

In [ ]:
# Full field — all 4 subfields visible at once
# Change map_key to "Q" or "U" to see polarisation maps
map_key = "T"
m   = mask_map(frames[ref_id][map_key], frames[ref_id][_wt(map_key)])
rms = float(np.std(m[m != hp.UNSEEN]))

show_map_full_field(
    m, vmin=-3*rms, vmax=3*rms,
    title=f"{ref_id}  —  {map_key}  (full field)",
    unit="Tcmb", cmap="coolwarm")

### 4.2 Single-map — matplotlib backend

In [ ]:
# Zoomed thumbnail — change rot to centre on a different patch
# Default: rot=(32, -51, 0) centres on ra0hdec-52.25 subfield
map_key = "T"
m   = mask_map(frames[ref_id][map_key], frames[ref_id][_wt(map_key)])
rms = float(np.std(m[m != hp.UNSEEN]))

show_map_thumbnail(
    m, vmin=-3*rms, vmax=3*rms,
    title=f"{ref_id}  —  {map_key}  (thumbnail)",
    unit="Tcmb", cmap="coolwarm")

### 4.3 Multi-frequency grids

3 × 2 layout: rows = 90 / 150 / 220 GHz, columns = Left / Right detector set.

In [ ]:
plotter.plot_grid(map_key="T", layout=layout, rot=rot_centre, suptitle="T maps")
plt.show()

In [ ]:
plotter.plot_grid(map_key="Q", layout=layout, rot=rot_centre, suptitle="Q maps")
plt.show()

In [ ]:
plotter.plot_grid(map_key="U", layout=layout, rot=rot_centre, suptitle="U maps")
plt.show()

## 5. E/B Decomposition

Diagnostic E and B maps from T/Q/U via `healpy.map2alm_lsq`.  
This is a quick inspection tool — not the final pure-B SPT pipeline.

> **Before running:** define `apod_mask` (load from file on the cluster).

In [ ]:
def apply_pol_mask(Q, U, obs_mask, apod_mask):
    """
    Apply observed mask + apodisation mask to Q/U.

    For harmonic transforms:
    - observed pixels get Q * apod_mask, U * apod_mask
    - unobserved pixels are set to 0
    """

    Q_clean = np.asarray(Q).copy()
    U_clean = np.asarray(U).copy()

    apod_mask = np.asarray(apod_mask)

    usable = obs_mask.copy()
    usable &= np.isfinite(apod_mask)
    usable &= apod_mask > 0

    Q_clean[usable] *= apod_mask[usable]
    U_clean[usable] *= apod_mask[usable]

    Q_clean[~usable] = 0.0
    U_clean[~usable] = 0.0

    return Q_clean, U_clean, usable

In [ ]:
# obs_pol   = TT > 0                           # simple observed-pixel mask
# apod_mask = np.load("/path/to/apod_mask.npy") # load on cluster

Q_apod, U_apod, usable_pol = apply_pol_mask(Q, U, obs_pol, apod_mask)
print("Usable pixels after apodisation:", usable_pol.sum())

In [ ]:
def make_EB_maps_lsq(
    T,
    Q,
    U,
    TT=None,
    QQ=None,
    UU=None,
    apod_mask=None,
    lmax=512,
    mmax=None,
    tol=1e-8,
    maxiter=20,
):
    """
    Make quick diagnostic E/B maps from T/Q/U using healpy.sphtfunc.map2alm_lsq.

    WARNING:
    This is useful for inspection/debugging.
    It is not a final pure-B SPT analysis.
    """

    T = np.asarray(T).copy()
    Q = np.asarray(Q).copy()
    U = np.asarray(U).copy()

    nside = hp.get_nside(T)

    if mmax is None:
        mmax = lmax

    # Observed mask
    obs = np.isfinite(T) & np.isfinite(Q) & np.isfinite(U)
    obs &= T != hp.UNSEEN
    obs &= Q != hp.UNSEEN
    obs &= U != hp.UNSEEN

    if TT is not None:
        obs &= np.isfinite(TT)
        obs &= TT > 0

    if QQ is not None:
        obs &= np.isfinite(QQ)
        obs &= QQ > 0

    if UU is not None:
        obs &= np.isfinite(UU)
        obs &= UU > 0

    # Apply apodisation if given
    if apod_mask is not None:
        apod_mask = np.asarray(apod_mask)
        obs &= np.isfinite(apod_mask)
        obs &= apod_mask > 0

        T[obs] *= apod_mask[obs]
        Q[obs] *= apod_mask[obs]
        U[obs] *= apod_mask[obs]

    # For harmonic transform, outside observed region must be numeric
    T[~obs] = 0.0
    Q[~obs] = 0.0
    U[~obs] = 0.0

    # Solve TQU -> TEB alms
    alms, residual_maps, rel_residual = hp.sphtfunc.map2alm_lsq(
        [T, Q, U],
        lmax=lmax,
        mmax=mmax,
        pol=True,
        tol=tol,
        maxiter=maxiter,
    )

    almT, almE, almB = alms

    # alm -> maps
    T_rec, E_map, B_map = hp.alm2map(
        [almT, almE, almB],
        nside=nside,
        lmax=lmax,
        mmax=mmax,
        pol=True,
        verbose=False,
    )

    # Hide unobserved pixels for plotting
    E_map[~obs] = hp.UNSEEN
    B_map[~obs] = hp.UNSEEN

    return E_map, B_map, almE, almB, residual_maps, rel_residual, obs

In [ ]:
E_map, B_map, almE, almB, residual_maps, rel_residual, eb_obs = make_EB_maps_lsq(
    T, Q, U,
    TT=TT, QQ=QQ, UU=UU,
    apod_mask=apod_mask,
    lmax=512,
    tol=1e-8,
    maxiter=20,
)
print("Relative residual:", rel_residual)

## 6. Polatm-Subtracted Maps

Load and plot maps from the **most-cleaned** map product:
`full_2019_p10_elnod_pair_difference_healpix_recent_cal_lpf3k_50_250_renormalized_polatm_subtracted`

Why this version:
- **polatm_subtracted** — atmospheric polarization contamination removed
- **elnod calibration** — more accurate than default calibration
- **renormalized** — filter transfer function already corrected
- **healpix** — already in HEALPix format, no conversion needed
- **lpf3k** — low-pass filtered at ℓ ~ 3000 (sets our lmax ceiling)

> **Cluster only.** Paths resolve on `scott`; cells will not run locally.

In [ ]:
# --- 6.1  Discover available obs IDs for the polatm-subtracted product ---

MAP_TYPE_POLATM = (
    "full_2019_p10_elnod_pair_difference_healpix_recent_cal_lpf3k"
    "_50_250_renormalized_polatm_subtracted"
)

polatm_dir = f"{base_bb2019}/{subfield}/{MAP_TYPE_POLATM}"
polatm_obsids = sorted(os.listdir(polatm_dir))

print(f"Subfield : {subfield}")
print(f"Map type : {MAP_TYPE_POLATM}")
print(f"Obs IDs  : {len(polatm_obsids)} found")
print("First 5  :", polatm_obsids[:5])
print("Last  5  :", polatm_obsids[-5:])

In [ ]:
# --- 6.2  Load one obs ID from the polatm-subtracted product ---
# Change polatm_obsid to inspect a different observation.

polatm_obsid = polatm_obsids[0]   # first available obs ID; change as needed

polatm_file = (
    f"{polatm_dir}/{polatm_obsid}"
    f"/{MAP_TYPE_POLATM}_{polatm_obsid}.g3"
)
print("Loading:", polatm_file)
print("Exists :", os.path.exists(polatm_file))

target_ids = [
    "Left90GHz",  "Right90GHz",
    "Left150GHz", "Right150GHz",
    "Left220GHz", "Right220GHz",
]

frames_polatm = {}
for fr in core.G3File(polatm_file):
    if fr.type != core.G3FrameType.Map:
        continue
    mid = fr["Id"]
    if mid not in target_ids:
        continue
    W = fr["Wpol"]
    frames_polatm[mid] = {
        "T":  np.asarray(fr["T"],  float),
        "Q":  np.asarray(fr["Q"],  float),
        "U":  np.asarray(fr["U"],  float),
        "TT": np.asarray(W.TT, float),
        "QQ": np.asarray(W.QQ, float),
        "UU": np.asarray(W.UU, float),
    }

print(f"Loaded {len(frames_polatm)} frames:", list(frames_polatm))

### 6.3  Plot T / Q / U maps (polatm-subtracted)

3 × 2 grid: rows = 90 / 150 / 220 GHz, columns = Left / Right detector set.

In [ ]:
# --- 6.3  Plot T / Q / U (polatm-subtracted) ---
# rot_centre and layout are reused from §3.

plotter_polatm = MapPlotter(frames_polatm)

layout = [
    ("Left90GHz",  "Right90GHz"),
    ("Left150GHz", "Right150GHz"),
    ("Left220GHz", "Right220GHz"),
]

for stokes in ("T", "Q", "U"):
    fig = plotter_polatm.plot_grid(
        map_key=stokes,
        layout=layout,
        rot=rot_centre,
        suptitle=f"{stokes} maps — polatm_subtracted  |  obsid {polatm_obsid}  |  {subfield}",
    )
    plt.show()